The below code sets up an ETH3D dataset for testing. If you choose to use a different dataset that has multiple image folders, you can specify those in the folders list.

In [1]:
import pycolmap

reconstruction = pycolmap.Reconstruction('../SfM_data_clean/courtyard/dslr_calibration_undistorted')

In [2]:
import os

image_list = []
img_folder = '../SfM_data_clean/courtyard/images'
folders = ['dslr_images_undistorted']
for folder in folders:
    subfolder = f'{img_folder}/{folder}'
    
    images = os.listdir(subfolder)
    for i in range(len(images)):
        image_list.append(images[i])

In [ ]:
import numpy as np

def rescale_calibration_matrix(K, original_size, new_size):
    K_new = K.copy()
    
    # Scaling
    s_x = new_size[0] / original_size[0]
    s_y = new_size[1] / original_size[1]
    
    K_new[0, 0] = K_new[0, 0] * s_y 
    K_new[1, 1] = K_new[1, 1] * s_x
    K_new[0, 2] = K_new[0, 2] * s_y 
    K_new[1, 2] = K_new[1, 2] * s_x
    
    return K_new

In [4]:
from tqdm import tqdm
import cv2

K_list = {}
img_dict = {}
img_size = {}
long_side = 2000

for i in tqdm(range(len(image_list))):
    img_id1 = image_list[i]
    
    img_1_num = 0
    img_1_folder = ""

    for folder in folders:

        fname1 = f'{folder}/{img_id1}'

        if img_1_num == 0:
            for k in range(1, len(reconstruction.images) + 1):
                if fname1 == reconstruction.images[k].name:
                    img_1_num = k
                    img_1_folder = folder
                if img_1_num != 0:
                    break

    fname1 = f'{img_folder}/{img_1_folder}/{img_id1}'
    
    K1 = np.eye(3)
    params = reconstruction.images[img_1_num].camera.params
    K1[0, 0], K1[1, 1], K1[0, 2], K1[1, 2] = params
    
    img1 = cv2.cvtColor(cv2.imread(fname1), cv2.COLOR_BGR2RGB)

    scale = long_side / max(img1.shape[:2])
    new_size = (round(img1.shape[0] * scale), round(img1.shape[1] * scale))
    K1 = rescale_calibration_matrix(K1, img1.shape[:2], new_size)
    
    img_dict[img_id1] = [fname1, img_1_num]
    img_size[img_id1] = new_size
    K_list[img_id1] = K1

100%|██████████| 38/38 [00:15<00:00,  2.50it/s]


In [5]:
scene_ids = []
pts0_list = []
pts1_list = []

sift = cv2.SIFT_create(nfeatures=8000)
features = {}
for img_id in tqdm(image_list):
    fname, img_num = img_dict[img_id]
    img = cv2.imread(fname, cv2.IMREAD_GRAYSCALE)
    h, w = img_size[img_id]
    img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
    features[img_id] = sift.detectAndCompute(img, None)

matcher = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=64))
ratio = 0.8

total_iterations = (len(image_list) - 1) * (len(image_list)) / 2
progress_bar = tqdm(total=total_iterations)

for i in range(0, len(image_list)):
    for j in range(i + 1, len(image_list)):
        img_id1 = image_list[i]
        img_id2 = image_list[j]
        scene_id = f'{img_id1}-{img_id2}'
        scene_ids.append(scene_id)

        kp1, des1 = features[img_id1]
        kp2, des2 = features[img_id2]

        knn = matcher.knnMatch(des1, des2, k=2)
        matches = [m for m, n in (pair for pair in knn if len(pair) == 2) if m.distance < ratio * n.distance]

        pts0, pts1 = list(), list()

        for k in range(len(matches)):
            pts0.append(kp1[matches[k].queryIdx].pt)
            pts1.append(kp2[matches[k].trainIdx].pt)

        pts0_list.append(np.array(pts0).reshape(-1, 2))
        pts1_list.append(np.array(pts1).reshape(-1, 2))
        progress_bar.update(1)

100%|██████████| 38/38 [00:26<00:00,  1.42it/s]
100%|██████████| 703/703.0 [05:19<00:00,  2.00it/s]

In [6]:
from scipy.spatial.transform import Rotation as R

def R_from_image(image):
    qvec = image.cam_from_world().rotation.quat
    rotation = R.from_quat(qvec)
    rotation_matrix = rotation.as_matrix()
    return rotation_matrix

In [7]:
K1_list = []
K2_list = []
dR_list = []
R2_list = []

for i in range(len(scene_ids)):
    scene = scene_ids[i]
    img_id1 = scene.split('-')[0]
    img_id2 = scene.split('-')[1]
    
    fname1, img_1_num = img_dict[img_id1]
    fname2, img_2_num = img_dict[img_id2]
    K1 = K_list[img_id1]
    K2 = K_list[img_id2]

    R1 = R_from_image(reconstruction.images[img_1_num])
    R2 = R_from_image(reconstruction.images[img_2_num])
    dR = np.dot(R2, R1.T)
    
    K1_list.append(K1)
    K2_list.append(K2)
    dR_list.append(dR)
    R2_list.append(R2)

In [ ]:
from helper_funcs import relpose, world_direction

key_list = list(img_dict.keys())
key_dict = dict(zip(key_list, range(len(key_list))))
adj_mat = np.zeros((len(image_list), len(image_list)))
t_dict = {}
min_inliers = 100
for i in tqdm(range(len(scene_ids))):
    scene = scene_ids[i]
    
    img_id1 = key_dict[scene.split('-')[0]]
    img_id2 = key_dict[scene.split('-')[1]]
    
    pts0 = pts0_list[i]
    pts1 = pts1_list[i]

    K1 = K1_list[i]
    K2 = K2_list[i]
    R2 = R2_list[i]

    if len(pts0) < min_inliers:
        continue

    # 5-point algorithm
    y0 = cv2.undistortPoints(pts0.reshape(-1, 1, 2), K1, None).reshape(-1, 2)
    y1 = cv2.undistortPoints(pts1.reshape(-1, 1, 2), K2, None).reshape(-1, 2)
    E, inl_mask = cv2.findEssentialMat(y0, y1, np.eye(3), cv2.USAC_ACCURATE, 0.999, 1.0 / K1[0, 0])
    if E is None or E.shape[0] < 3 or inl_mask.sum() < min_inliers:
        continue
    R, t = relpose(E[:3], y0, y1, np.eye(3), np.eye(3), mask=inl_mask)

    adj_mat[img_id1][img_id2] = 1
    adj_mat[img_id2][img_id1] = 1


    gamma_ij = world_direction(R2, t)
        
    t_dict[(img_id1, img_id2)] = gamma_ij
    t_dict[(img_id2, img_id1)] = -gamma_ij

from scipy.sparse.csgraph import connected_components
n_components, _ = connected_components(adj_mat)
print(f"Edges kept: {int(adj_mat.sum() // 2)} of {len(scene_ids)} pairs; connected components: {n_components}")

100%|██████████| 703/703 [01:04<00:00, 10.97it/s]


Edges kept: 290 of 703 pairs; connected components: 1


In [9]:
from scipy.spatial.transform import Rotation as R

t_true_dict = {}
for image_id, image in reconstruction.images.items():
    t_true_dict[image.name] = image.projection_center()

t_true = np.zeros((3, len(images)))
for i in range(len(images)):
    img_id = key_dict[list(t_true_dict.keys())[i].split("/")[1]]
    value = list(t_true_dict.values())[i]
    t_true[:, img_id] = value

In [ ]:
from utils.indexing import tijmat_from_dict

tij_mat = tijmat_from_dict(adj_mat, t_dict)

The below code generates testing results on the dataset for cycle sync

In [11]:
from cycle_sync import Cycle_Sync
from test_case import align_procrustes

params = {}
t_est, out = Cycle_Sync(adj_mat, tij_mat, params)

print("Estimated camera locations:")
print(t_est, "\n")

print("Alpha (Edge Scales):")
print(out.alph, "\n")

print("Total time taken:")
print(out.TotalTime, "seconds \n")

X_aligned, R, scale, translation, err = align_procrustes(t_est, t_true)

print("Scale:")
print(scale, "\n")

print("Translation Vector:")
print(translation, "\n")

print("Aligned estimate:")
print(X_aligned, "\n")

print("RMS alignment error:", err)
print("Median camera error:", np.median(np.linalg.norm(X_aligned - t_true, axis=0)))

Estimated camera locations:
[[-3.09426917e+00 -3.21939485e+00 -3.36029818e+00 -3.43792533e+00
  -3.62112713e+00 -3.81598709e+00 -3.97146542e+00 -4.08116574e+00
  -4.14826322e+00 -4.22163420e+00 -4.09017267e+00 -3.44159943e+00
  -3.02096934e+00 -6.95283081e+00 -5.95897178e+00 -5.35842307e+00
  -6.34198687e+00 -5.61501547e+00 -4.89718462e+00 -4.47338214e+00
  -4.57648787e+00 -4.93428231e+00 -1.84775614e+00 -2.40094579e+00
  -3.07738250e+00  8.98397565e+00  7.49710676e+00  6.80251206e+00
   6.87309734e+00  7.92505846e+00  1.03108043e+01  1.01704791e+01
   5.21224616e+00  1.78845811e+00  4.97852230e+00  8.73702144e+00
   1.18370954e+01  1.28425442e+01]
 [-3.89156422e+00 -2.91074675e+00 -2.06237814e+00 -1.28147679e+00
  -3.45632335e-01  4.46579804e-01  1.28674788e+00  2.27800882e+00
   3.15976525e+00  4.03445378e+00  5.07075782e+00  5.72479417e+00
  -6.62493486e-01  6.70215978e+00  5.97602528e+00  4.53492960e+00
   4.46324780e+00  2.50247009e+00  5.52160491e-01 -1.66249770e+00
  -2.67028848

In [12]:
def robust_rel_err(t):
    w = np.ones(t.shape[1])
    for _ in range(200):
        mu_est = (t * w).sum(axis=1, keepdims=True) / w.sum()
        mu_true = (t_true * w).sum(axis=1, keepdims=True) / w.sum()
        Xc, Yc = t - mu_est, t_true - mu_true
        scale = np.sum(w * np.sum(Xc * Yc, axis=0)) / np.sum(w * np.sum(Xc * Xc, axis=0))
        r = np.linalg.norm(scale * Xc - Yc, axis=0)
        w = 1 / np.maximum(r, 1e-9)
    radius = np.median(np.linalg.norm(t_true - np.median(t_true, axis=1, keepdims=True), axis=0))
    e = r / radius
    return np.median(e), 100 * np.mean(e < 0.1)

t_base, _ = Cycle_Sync(adj_mat, tij_mat, {'tau1': 0, 'WLSiters': 1})
print("Median error / scene radius (robust alignment), and % of cameras within 0.1:")
print("Cycle-Sync:                  %.4f  (%.0f%%)" % robust_rel_err(t_est))
print("Plain least squares baseline: %.4f  (%.0f%%)" % robust_rel_err(t_base))

Median error / scene radius (robust alignment), and % of cameras within 0.1:
Cycle-Sync:                  0.0121  (71%)
Plain least squares baseline: 0.7393  (3%)
